In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
PROJECT_DIR = "/content/drive/MyDrive/dataset03"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/train/real", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/train/fake", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/validation/real", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/validation/fake", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/test/real", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/test/fake", exist_ok=True)
print("Folders created at:", PROJECT_DIR)

Folders created at: /content/drive/MyDrive/dataset03


In [4]:
%cd /content/drive/MyDrive/dataset03
!pip install -q albumentations

/content/drive/MyDrive/dataset03


In [5]:
%%writefile preprocessing.py
"""
preprocessing.py
-----------------
Handles image transforms and DataLoader creation for the
Real vs Fake medical image classifier (Member 1).

Folder structure expected:

dataset03/
    train/
        real/
        fake/
    validation/
        real/
        fake/
    test/
        real/
        fake/
"""

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transforms():
    """Augmentation ONLY for training data. Kept mild so medical meaning isn't distorted.
    Lambda forces RGB so grayscale Kaggle scans (1-channel) don't crash DenseNet (expects 3)."""
    return transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_eval_transforms():
    """No augmentation for validation/test — just RGB-convert, resize + normalize.
    This exact same transform is reused by predict.py at inference time, so a
    doctor/user uploading any size/format scan never needs to resize manually."""
    return transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_dataloaders(data_dir="dataset03", batch_size=32, num_workers=2):
    """
    Builds train/validation/test DataLoaders using ImageFolder.
    ImageFolder automatically assigns class indices alphabetically:
        fake -> 0, real -> 1   (folder names are sorted alphabetically)
    We save this mapping explicitly so predict.py stays consistent.
    """
    train_dir = os.path.join(data_dir, "train")
    val_dir = os.path.join(data_dir, "validation")
    test_dir = os.path.join(data_dir, "test")

    train_dataset = datasets.ImageFolder(train_dir, transform=get_train_transforms())
    val_dataset = datasets.ImageFolder(val_dir, transform=get_eval_transforms())
    test_dataset = datasets.ImageFolder(test_dir, transform=get_eval_transforms())

    assert train_dataset.class_to_idx == val_dataset.class_to_idx == test_dataset.class_to_idx, \
        "Mismatch in class folders between train/validation/test!"

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, train_dataset.class_to_idx


if __name__ == "__main__":
    train_loader, val_loader, test_loader, class_to_idx = get_dataloaders()
    print("Class mapping:", class_to_idx)
    print("Train batches:", len(train_loader))
    print("Val batches:", len(val_loader))
    print("Test batches:", len(test_loader))

Overwriting preprocessing.py


In [6]:
%%writefile train.py
"""
train.py
--------
Trains a DenseNet121 (transfer learning) to classify medical images
as Real or Fake.

Usage:
    python train.py --data_dir dataset --epochs 25 --batch_size 32
"""

import argparse
import json
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import matplotlib.pyplot as plt

from preprocessing import get_dataloaders


def build_model(num_classes=2, freeze_backbone=True):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default="dataset")
    parser.add_argument("--epochs", type=int, default=25)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--unfreeze_after", type=int, default=8,
                         help="Epoch after which backbone layers are unfrozen for fine-tuning")
    parser.add_argument("--output_dir", default="models")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_loader, val_loader, test_loader, class_to_idx = get_dataloaders(
        args.data_dir, batch_size=args.batch_size
    )

    idx_to_class = {str(v): k.capitalize() for k, v in class_to_idx.items()}
    with open(os.path.join(args.output_dir, "class_names.json"), "w") as f:
        json.dump(idx_to_class, f, indent=2)
    print("Class mapping:", idx_to_class)

    model = build_model(num_classes=len(class_to_idx), freeze_backbone=True).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_model_path = os.path.join(args.output_dir, "cnn_model.pth")

    for epoch in range(args.epochs):
        start = time.time()

        if epoch == args.unfreeze_after:
            print(">> Unfreezing backbone for fine-tuning")
            for param in model.features.parameters():
                param.requires_grad = True
            optimizer = optim.Adam(model.parameters(), lr=args.lr / 10)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_acc)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - start
        print(f"Epoch {epoch+1}/{args.epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {elapsed:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_to_idx": class_to_idx,
                "val_acc": val_acc,
            }, best_model_path)
            print(f"  -> New best model saved (val_acc={val_acc:.4f})")

    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    print(f"Model saved to: {best_model_path}")

    plt.figure()
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.title("Training vs Validation Accuracy")
    plt.savefig(os.path.join(args.output_dir, "training_accuracy.png"))

    plt.figure()
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.title("Training vs Validation Loss")
    plt.savefig(os.path.join(args.output_dir, "training_loss.png"))

    print("Saved training_accuracy.png and training_loss.png")


if __name__ == "__main__":
    main()

Overwriting train.py


In [7]:
%%writefile evaluate.py
"""
evaluate.py
-----------
Loads the trained cnn_model.pth and evaluates it on the held-out test set.

Usage:
    python evaluate.py --data_dir dataset --model_path models/cnn_model.pth
"""

import argparse
import json
import os

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)

from preprocessing import get_dataloaders
from train import build_model


@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default="dataset")
    parser.add_argument("--model_path", default="models/cnn_model.pth")
    parser.add_argument("--output_dir", default="models")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    _, _, test_loader, class_to_idx = get_dataloaders(args.data_dir)

    checkpoint = torch.load(args.model_path, map_location=device)
    model = build_model(num_classes=len(class_to_idx)).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    with open(os.path.join(args.output_dir, "class_names.json")) as f:
        idx_to_class = json.load(f)
    class_names = [idx_to_class[str(i)] for i in range(len(idx_to_class))]

    labels, preds, probs = get_predictions(model, test_loader, device)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")

    cm = confusion_matrix(labels, preds)

    report = (
        f"Test Accuracy : {acc:.4f}\n"
        f"Precision     : {prec:.4f}\n"
        f"Recall        : {rec:.4f}\n"
        f"F1 Score      : {f1:.4f}\n"
        f"ROC-AUC       : {auc:.4f}\n\n"
        f"Confusion Matrix (rows=actual, cols=predicted):\n{cm}\n"
        f"Class order: {class_names}\n"
    )
    print(report)

    with open(os.path.join(args.output_dir, "metrics.txt"), "w") as f:
        f.write(report)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap="Blues")
    plt.title("Confusion Matrix - Test Set")
    plt.savefig(os.path.join(args.output_dir, "confusion_matrix.png"))
    print("Saved metrics.txt and confusion_matrix.png")


if __name__ == "__main__":
    main()

Overwriting evaluate.py


In [ ]:
%%writefile predict.py
"""
predict.py
----------
Inference script/function for a single MRI/CT/X-ray image.
This is the function Member 3 (backend) will import and call from FastAPI.
"""

import argparse
import json
import os

import torch
from PIL import Image

from preprocessing import get_eval_transforms
from train import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_model(model_path="models/cnn_model.pth", class_names_path=None):
    if class_names_path is None:
        class_names_path = os.path.join(os.path.dirname(model_path), "class_names.json")

    checkpoint = torch.load(model_path, map_location=DEVICE)
    class_to_idx = checkpoint["class_to_idx"]

    model = build_model(num_classes=len(class_to_idx))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    with open(class_names_path) as f:
        idx_to_class = json.load(f)

    return model, idx_to_class


def predict_image(model, idx_to_class, image_path):
    transform = get_eval_transforms()
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        confidence, pred_idx = torch.max(probs, dim=0)

    label = idx_to_class[str(pred_idx.item())]
    confidence_pct = round(confidence.item() * 100, 2)

    return {
        "prediction": label,
        "confidence": confidence_pct,
        "class": label,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", required=True)
    parser.add_argument("--model_path", default="models/cnn_model.pth")
    args = parser.parse_args()

    model, idx_to_class = load_model(args.model_path)
    result = predict_image(model, idx_to_class, args.image)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

Overwriting predict.py


In [ ]:
%%writefile clean_dataset.py
"""
clean_dataset.py
-----------------
Run once after organizing Kaggle images into dataset/train|validation|test/real|fake,
before running train.py. Removes corrupted/unreadable files; does NOT resize anything.

Usage:
    python clean_dataset.py --data_dir dataset
"""

import argparse
import os
from collections import Counter
from PIL import Image, UnidentifiedImageError

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def scan_folder(folder_path):
    removed = 0
    kept = 0
    sizes = Counter()
    formats = Counter()

    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        ext = os.path.splitext(fname)[1].lower()

        if ext not in VALID_EXTENSIONS:
            print(f"  [SKIP FORMAT] {fpath} (unsupported extension: {ext})")
            os.remove(fpath)
            removed += 1
            continue

        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                sizes[img.size] += 1
                formats[img.format] += 1
                kept += 1
        except (UnidentifiedImageError, OSError) as e:
            print(f"  [CORRUPTED] {fpath} -> removing ({e})")
            os.remove(fpath)
            removed += 1

    return kept, removed, sizes, formats


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", default="dataset")
    args = parser.parse_args()

    total_kept, total_removed = 0, 0

    for split in ["train", "validation", "test"]:
        for cls in ["real", "fake"]:
            folder = os.path.join(args.data_dir, split, cls)
            if not os.path.isdir(folder):
                print(f"[MISSING FOLDER] {folder} — skipping")
                continue

            print(f"\nScanning {folder} ...")
            kept, removed, sizes, formats = scan_folder(folder)
            total_kept += kept
            total_removed += removed

            print(f"  Kept: {kept} | Removed: {removed}")
            if formats:
                print(f"  Formats found: {dict(formats)}")
            if sizes:
                print(f"  Most common original sizes: {sizes.most_common(3)}")
                print("  (No action needed — auto-resizes to 224x224 during training/inference)")

    print(f"\n=== Done. Total kept: {total_kept} | Total removed: {total_removed} ===")


if __name__ == "__main__":
    main()

Overwriting clean_dataset.py


In [ ]:
import os
for split in ["train", "validation", "test"]:
    for cls in ["real", "fake"]:
        path = f"{PROJECT_DIR}/{split}/{cls}"
        count = len(os.listdir(path))
        print(f"{split}/{cls}: {count} images")

train/real: 700 images
train/fake: 700 images
validation/real: 150 images
validation/fake: 150 images
test/real: 150 images
test/fake: 150 images


In [ ]:
!python clean_dataset.py --data_dir .


Scanning ./train/real ...
  Kept: 700 | Removed: 0
  Formats found: {'PNG': 351, 'JPEG': 349}
  Most common original sizes: [((299, 299), 350), ((512, 512), 174), ((225, 225), 36)]
  (No action needed — auto-resizes to 224x224 during training/inference)

Scanning ./train/fake ...
  Kept: 700 | Removed: 0
  Formats found: {'PNG': 700}
  Most common original sizes: [((224, 224), 700)]
  (No action needed — auto-resizes to 224x224 during training/inference)

Scanning ./validation/real ...
  Kept: 150 | Removed: 0
  Formats found: {'JPEG': 75, 'PNG': 75}
  Most common original sizes: [((299, 299), 75), ((512, 512), 39), ((225, 225), 13)]
  (No action needed — auto-resizes to 224x224 during training/inference)

Scanning ./validation/fake ...
  Kept: 150 | Removed: 0
  Formats found: {'PNG': 150}
  Most common original sizes: [((224, 224), 150)]
  (No action needed — auto-resizes to 224x224 during training/inference)

Scanning ./test/real ...
  Kept: 150 | Removed: 0
  Formats found: {'JPEG

In [ ]:
!python train.py --data_dir . --epochs 25 --batch_size 32 --unfreeze_after 8

Using device: cuda
Class mapping: {'0': 'Fake', '1': 'Fake_synthetic', '2': 'Real'}
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:00<00:00, 161MB/s]
Epoch 1/25 | train_loss=0.9995 train_acc=0.5002 | val_loss=0.9215 val_acc=0.5550 | 70.9s
  -> New best model saved (val_acc=0.5550)
Epoch 2/25 | train_loss=0.8668 train_acc=0.6410 | val_loss=0.8145 val_acc=0.6933 | 23.3s
  -> New best model saved (val_acc=0.6933)
Epoch 3/25 | train_loss=0.7725 train_acc=0.7186 | val_loss=0.7415 val_acc=0.7283 | 25.5s
  -> New best model saved (val_acc=0.7283)
Epoch 4/25 | train_loss=0.7133 train_acc=0.7365 | val_loss=0.6814 val_acc=0.7450 | 24.3s
  -> New best model saved (val_acc=0.7450)
Epoch 5/25 | train_loss=0.6745 train_acc=0.7376 | val_loss=0.6436 val_acc=0.7467 | 24.1s
  -> New best model saved (val_acc=0.7467)
Epoch 6/25 | train_loss=0.6357 train_acc=0.7462 | val_loss=0.6344 val_acc=0.7

In [ ]:

!python evaluate.py --data_dir . --model_path models/cnn_model.pth

Traceback (most recent call last):
  File "/content/drive/MyDrive/dataset02/evaluate.py", line 98, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/dataset02/evaluate.py", line 66, in main
    prec = precision_score(labels, preds)
  File "/usr/local/lib/python3.13/dist-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py", line 2247, in precision_score
    p, _, _, _ = precision_recall_fscore_support(
                 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        y_true,
        ^^^^^^^
    ...<6 lines>...
        zero_division=zero_division,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/utils/_param_validation.py", line 189, in wrapper
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py", line 1830, in precision_recall_fscore

In [ ]:
!python predict.py --image test/fake/<some_image_filename>.jpg --model_path models/cnn_model.pth

/bin/bash: line 1: some_image_filename: No such file or directory


In [ ]:
!ls -la models/

total 27859
drwx------ 2 root root     4096 Sep 15 16:09 .
drwx------ 8 root root     4096 Sep 15 15:54 ..
-rw------- 1 root root       57 Sep 15 15:55 class_names.json
-rw------- 1 root root 28434315 Sep 15 16:09 cnn_model.pth
-rw------- 1 root root    20804 Aug 24 13:40 confusion_matrix.png
-rw------- 1 root root      217 Aug 24 13:40 metrics.txt
-rw------- 1 root root    31912 Sep 15 16:09 training_accuracy.png
-rw------- 1 root root    30231 Sep 15 16:09 training_loss.png
